This notebook aggregates the individual taxi trips to a hourly and location dataset to enable the predicition of demand of taxis at a specific location.

In [ ]:
import polars as pl
import pandas as pd
import h3

In [6]:
data = pl.read_parquet("../data/02_merged_data.parquet")
display(data.shape)
display(data.head())

(12727858, 60)

trip_id,taxi_id,trip_start,trip_end,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,fare_usd,tips_usd,tolls_usd,extras_usd,trip_total_usd,payment_type,company,pickup_lat,pickup_lon,dropoff_lat,dropoff_lon,average_speed_mph,hour,day_of_week,month,is_weekend,bin_30min,bin_1h,bin_4h,bin_1d,bin_1w,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,date_day,DayOfWeek_Friday,DayOfWeek_Saturday,DayOfWeek_Sunday,DayOfWeek_Thursday,DayOfWeek_Tuesday,DayOfWeek_Wednesday,BankHoliday,CorporateHoliday,CityHoliday,StateHoliday,temperature_2m,rain,precipitation,wind_speed_10m,wind_direction_10m,wind_gusts_10m,apparent_temperature,relative_humidity_2m,snow_depth,snowfall,temperature_2m_max,temperature_2m_min
str,str,"datetime[ms, America/Chicago]","datetime[ms, America/Chicago]",duration[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,i8,i8,i8,bool,"datetime[ms, America/Chicago]","datetime[ms, America/Chicago]","datetime[ms, America/Chicago]","datetime[ms, America/Chicago]","datetime[ms, America/Chicago]",f64,f64,f64,f64,f64,f64,date,u8,u8,u8,u8,u8,u8,i8,i8,i8,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""ff4d916013cd671c8ec9dcb181a3ef…","""0ccfc1f0f221ef68515db370201623…",2024-01-01 00:45:00 CST,2024-01-01 01:07:00 CST,22m,15.0,1.7032e10,1.7031e10,76.0,8.0,38.25,8.55,0.0,4.0,50.8,"""credit card""","""choice taxi association""",41.980264,-87.913625,41.899602,-87.633308,40.909091,0,1,1,false,2024-01-01 00:30:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,0.0,1.0,0.781831,0.62349,0.5,0.866025,2024-01-01,0,0,0,0,0,0,1,1,1,1,0.35,0.0,0.1,30.449274,335.556061,44.279999,-6.48454,85.168861,0.01,0.07,2.5,-4.6
"""031c6a339f8b92f0cd71d93503c1f4…","""0fdab9be71f6d88e3d3a2e115afc5a…",2024-01-01 00:45:00 CST,2024-01-01 01:31:59 CST,46m 59s,5.94,1.7031e10,1.7032e10,32.0,3.0,26.75,0.0,0.0,1.0,28.25,"""credit card""","""city service""",41.878866,-87.625192,41.965812,-87.655879,7.585669,0,1,1,false,2024-01-01 00:30:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,0.0,1.0,0.781831,0.62349,0.5,0.866025,2024-01-01,0,0,0,0,0,0,1,1,1,1,0.35,0.0,0.1,30.449274,335.556061,44.279999,-6.48454,85.168861,0.01,0.07,2.5,-4.6
"""0852ba909d47fda572bf5cfd6f7534…","""50e6045759886233d0e834d9877f52…",2024-01-01 00:45:00 CST,2024-01-01 01:41:00 CST,56m,0.8,1.7032e10,1.7031e10,56.0,32.0,44.25,0.0,0.0,6.0,50.25,"""cash""","""taxi affiliation services""",41.785999,-87.750934,41.884987,-87.620993,0.857143,0,1,1,false,2024-01-01 00:30:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,0.0,1.0,0.781831,0.62349,0.5,0.866025,2024-01-01,0,0,0,0,0,0,1,1,1,1,0.35,0.0,0.1,30.449274,335.556061,44.279999,-6.48454,85.168861,0.01,0.07,2.5,-4.6
"""1181a567553667587f1df4d77e84a0…","""b56b31883524426035c290bfdbe4e0…",2024-01-01 00:45:00 CST,2024-01-01 01:10:04 CST,25m 4s,5.55,1.7031e10,1.7031e10,6.0,24.0,18.25,3.95,0.0,1.0,23.7,"""credit card""","""sun taxi""",41.944227,-87.655998,41.901207,-87.676356,13.284574,0,1,1,false,2024-01-01 00:30:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,0.0,1.0,0.781831,0.62349,0.5,0.866025,2024-01-01,0,0,0,0,0,0,1,1,1,1,0.35,0.0,0.1,30.449274,335.556061,44.279999,-6.48454,85.168861,0.01,0.07,2.5,-4.6
"""11945be772d77429461147177bc323…","""3835dded237bfc6b12d2181ec37148…",2024-01-01 00:45:00 CST,2024-01-01 01:04:00 CST,19m,2.16,1.7031e10,1.7031e10,8.0,28.0,11.25,0.0,0.0,1.0,12.75,"""credit card""","""blue ribbon taxi association""",41.899156,-87.626211,41.879255,-87.642649,6.821053,0,1,1,false,2024-01-01 00:30:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,2024-01-01 00:00:00 CST,0.0,1.0,0.781831,0.62349,0.5,0.866025,2024-01-01,0,0,0,0,0,0,1,1,1,1,0.35,0.0,0.1,30.449274,335.556061,44.279999,-6.48454,85.168861,0.01,0.07,2.5,-

# Cleaning
The data still contains columsn which were relevant for the temporal-spatial analysis but cannot be used for the prediction of demand.

In [ ]:
data.drop('trip_id', 'taxi_id', 'trip_end', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'dropoff_lat', 'dropoff_lon', inplace=True)

In [ ]:
df = data.to_pandas()


In [ ]:
def to_h3_cell(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None
    return h3.latlng_to_cell(float(lat), float(lon), 9)

df["start_h3_r9"] = [to_h3_cell(lat, lon) for lat, lon in zip(df["pickup_lat"],  df["pickup_lon"])]
df["end_h3_r9"]   = [to_h3_cell(lat, lon) for lat, lon in zip(df["dropoff_lat"], df["dropoff_lon"])]

print(f"  R9: {df['start_h3_r9'].nunique():>4} pickup cells, {df['end_h3_r9'].nunique():>4} dropoff cells")

In [ ]:
df.columns

In [ ]:
df.to_parquet("../data/03_merged_data_with_h3.parquet", index=False)

# normalisation